## Methodology
### Load docs --> Split docs (Create Chunks) --> Embed Chunks (Vector Embedding) --> Store in Vector DB --> Query using similarity search --> Document chain --> Retrieval Chain --> Get Response from LLM

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['USER_AGENT'] = os.getenv('USER_AGENT')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [2]:
# Data Ingetion (Scrapping data from website)

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/billing")
loader

In [ ]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/billing', 'title': 'Manage billing in your account - Docs by LangChain', 'language': 'en'}, page_content='Manage billing in your account - Docs by LangChainOur new LangChain Academy Course Deep Research with LangGraph is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KLangSmithPlatform for LLM observability and evaluationSetupOverviewCreate an account and API keySet up a workspaceManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesFAQsTroubleshootingCloud architecture and scalabilityRegions FAQAuthentication methodsData purging for complianceRelease versionsOur new LangChain Academy Course Deep Research with LangGraph is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KAsk AIGitHubForumForumSearch...NavigationSetupManage billing in your accountGet startedObservabilityEvaluationPrompt engineeringSelf-hostingAdministrationGet st

In [4]:
# Split docs (Create Chunks)

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

documents = text_splitter.split_documents(docs)

documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/billing', 'title': 'Manage billing in your account - Docs by LangChain', 'language': 'en'}, page_content='Manage billing in your account - Docs by LangChainOur new LangChain Academy Course Deep Research with LangGraph is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KLangSmithPlatform for LLM observability and evaluationSetupOverviewCreate an account and API keySet up a workspaceManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesFAQsTroubleshootingCloud architecture and scalabilityRegions FAQAuthentication methodsData purging for complianceRelease versionsOur new LangChain Academy Course Deep Research with LangGraph is now live! Enroll for free.Docs by LangChain home pagePythonSearch...⌘KAsk AIGitHubForumForumSearch...NavigationSetupManage billing in your accountGet startedObservabilityEvaluationPrompt engineeringSelf-hostingAdministrationGet st

In [5]:
# Embed Chunks (Vector Embedding)

from langchain_community.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

C:\Users\dharm\AppData\Local\Temp\ipykernel_10252\4210435144.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [6]:
# Store in Vector DB

from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(documents, embeddings)
vectorstoredb

In [7]:
# Query using similarity search

query = "Langsmith has two usage limits: total traces and extended"

results = vectorstoredb.similarity_search(query)

print(results[0].page_content)

​Optimization 2: limit usage
In the previous section, you managed data retention settings to optimize existing spend. In this section, you will use usage limits to prevent future overspend.
LangSmith has two usage limits: total traces and extended retention traces. These correspond to the two metrics tracked on the usage graph. You can use these in tandem to have granular control over spend.
To set limits, navigate back to Settings -> Usage and Billing -> Usage configuration. There is a table at the bottom of the page that lets you set usage limits per workspace. For each workspace, the two limits appear, along with a cost estimate:

Start by setting limits on production usage, since that is where the majority of spend comes from.
​Set a good total traces limit
Picking the right total traces limit depends on the expected load of traces that you will send to LangSmith. It is important to consider potential growth before setting a limit. For example:


In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002547D4B49B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002547D4B51C0>, root_client=<openai.OpenAI object at 0x000002547D419D30>, root_async_client=<openai.AsyncOpenAI object at 0x000002547D4B4980>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [9]:
# Document Chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    """Answer the following question based on the context provided:
    <context>
    {context}
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Answer the following question based on the context provided:\n    <context>\n    {context}\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002547D4B49B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002547D4B51C0>, root_client=<openai.OpenAI object at 0x000002547D419D30>, root_async_client=<openai.AsyncOpenAI object at 0x000002547D4B4980>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_nam

In [10]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": query,
    "context": [Document(page_content=results[0].page_content)]
    })

"Sure, here's an approach to setting a good total traces limit based on expected load and potential growth:\n\n1. **Assess Current Usage**: Start by analyzing the current number of traces that your production environment is generating. This can be done by reviewing historical data and identifying peak usage patterns.\n\n2. **Estimate Future Growth**: Consider factors that may contribute to an increase in trace usage, such as anticipated user growth, upcoming feature releases, or other business activities that could increase traffic and subsequently, tracing.\n\n3. **Set a Baseline Limit**: Based on your current usage and estimated growth, set a baseline limit that accommodates the current load and provides a buffer for anticipated increases. This limit should be high enough to prevent interruptions in service due to exceeded limits but low enough to keep spending under control.\n\n4. **Monitor and Adjust**: After setting the limit, continuously monitor usage trends and adjust the limit

In [11]:
# Retrieval Chain

retriever = vectorstoredb.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)

retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002546A809CD0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Answer the following question based on the context provided:\n    <context>\n    {context}\n    '), additional_kwargs={})])
            | ChatOpenAI(

In [12]:
# Get response from LLM

response = retrieval_chain.invoke({"input": query})

print(response['answer'])

How can you set and optimize usage limits in LangSmith to prevent future overspend? 

To set and optimize usage limits in LangSmith and prevent future overspend, follow these steps:

1. **Navigate to Usage Configuration**: Go to Settings -> Usage and Billing -> Usage configuration. At the bottom of the page, you will find a table where you can set usage limits per workspace.

2. **Set Limits for Production**: Start by setting limits on production usage since this is where the majority of the spend occurs. 

3. **Determine a Good Total Traces Limit**: Choose a total traces limit based on the expected load of traces you plan to send to LangSmith, accounting for potential growth.

4. **Understand Current Usage**: Use the Usage graph and Invoices to understand your current usage. The Usage graph, found under Settings -> Usage and Billing -> Usage Graph, shows how much of each usage-based pricing metric you have consumed. It breaks down charges by "tenant_id" (i.e., workspace ID) allowing y